In [2]:
# General imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Dataset imports
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from torch.utils.data import random_split
from PIL import Image

# Utilities from datautils file
from datautils import full_image_stats

# Imports to deal with files and directories
import os
import json


In [3]:
# Turn API key into json for kaggle readability
my_info = {
    "username": "MohMathHorse",
    "key": "KGAT_0e35e6587a4c574cf3b2c9c8bcb790a2"
}

os.makedirs('/root/.kaggle', exist_ok=True)

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(my_info, f)

os.chmod('/root/.kaggle/kaggle.json', 600)

print("Password setup complete!")

Password setup complete!


In [4]:
# Download database zip file
!kaggle datasets download -d moritzm00/utkface-cropped

Dataset URL: https://www.kaggle.com/datasets/moritzm00/utkface-cropped
License(s): unknown
100% 116M/116M [00:01<00:00, 104MB/s] 



In [5]:
# Unzip
!unzip -q utkface-cropped.zip -d ./facial_data

In [6]:
# Device selection
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using device: MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

Using device: CUDA


In [7]:
mean, std = full_image_stats('/content/facial_data/UTKFace')

In [8]:
''' Define transforms and datasets '''

# Define train transformations
train_transform = transforms.Compose([
    transforms.RandomRotation(degrees=15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Resize(256),
    transforms.CenterCrop(240),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])
# Define validation transformations
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(240),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Define dataset class
class FacesDataset(Dataset):
  def __init__(self, directory):
    super(FacesDataset, self).__init__()
    self.directory = directory
    self.imglist = os.listdir(self.directory)

  def __len__(self):
    return len(self.imglist)

  def __getitem__(self, idx):
    filename = self.imglist[idx]
    full_path = os.path.join(self.directory, filename)
    image = train_transform(Image.open(full_path).convert("RGB"))
    age = torch.tensor([float((filename.split('_'))[0])])
    return image, age

facial_dataset = FacesDataset('/content/facial_data/UTKFace')

In [9]:
''' Split and load data '''

# Define dataset splitting
def dataset_split(trainsize):
  train_dataset, val_dataset = random_split(facial_dataset, [trainsize, 1-trainsize])
  return train_dataset, val_dataset

# Split the dataset
train_dataset, val_dataset = dataset_split(0.8)

# Define batches from datasets
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


In [10]:
''' Model Architecture '''

# Convolutional block, contains conv2d block and other features, including batch norm, relu, and maxpool. This is to have a complete convolutional block for the full CNN
class convblock(nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size=3, padding=1, dropout_prob=0.0):
    super(convblock, self).__init__()
    self.block = nn.Sequential(
        nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, padding=padding),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    if dropout_prob > 0:
        self.dropout = nn.Dropout2d(p=dropout_prob)
    else:
        self.dropout = None

  def forward(self, x):
    x = self.block(x)
    if self.dropout is not None:
        x = self.dropout(x)
    return x

# Full CNN
class fullcnn(nn.Module):
  def __init__(self):
    super(fullcnn, self).__init__()
    self.convblock1 = convblock(3, 32, dropout_prob=0.5)
    self.convblock2 = convblock(32, 64)
    self.convblock3 = convblock(64, 128, dropout_prob=0.5)
    self.regressor = nn.Sequential(
        nn.Flatten(),
        nn.BatchNorm1d(128*30*30),
        nn.Linear(128*30*30, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.6),
        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.6),
        nn.Linear(512, 1)
    )
  def forward(self, x):
    x = self.convblock1(x)
    x = self.convblock2(x)
    x = self.convblock3(x)
    x = self.regressor(x)
    return x

# Create an instance of the model
FacesCNN = fullcnn()

In [11]:
''' Training Process '''

# Loss and optimizer
lossfunction = nn.HuberLoss(delta=5.0)
metrics_loss = nn.L1Loss()
optimizer = optim.Adam(FacesCNN.parameters(), lr=0.0005, weight_decay=0.05)

# Training epoch
def train_epoch(model, train_loader, loss_function, optimizer, device):
  model.train()
  running_loss = 0.0
  for images, age in train_loader:
    images, age = images.to(device), age.to(device)
    optimizer.zero_grad()
    outputs = model(images)
    loss = loss_function(outputs, age)
    loss.backward()
    optimizer.step()
    eval_loss = metrics_loss(outputs, age)
    running_loss += eval_loss.item() * images.size(0)
  epoch_loss = running_loss / len(train_loader.dataset)
  return epoch_loss

# Training loop
def training_loop(model, train_loader, val_loader, loss_function, optimizer, device, num_epochs):
  model.to(device)
  print('---Training Started---')
  for epoch in range(num_epochs):
    epoch_loss = train_epoch(model, train_loader, loss_function, optimizer, device)
    val_running_loss = 0.0
    # Set model to evaluation mode for validation
    model.eval()
    with torch.no_grad():
      for images, age in val_loader:
        images, age = images.to(device), age.to(device)
        outputs = model(images)
        loss = metrics_loss(outputs, age)
        val_running_loss += loss.item() * images.size(0)
    val_loss = val_running_loss / len(val_loader.dataset)
    print(f'Epoch {epoch}: Training loss is {epoch_loss}, validation loss is {val_loss}.')
  print('---Training Over---')
  return model

In [12]:
# Train the model!
Trained_Model = training_loop(
    FacesCNN,
    train_loader,
    val_loader,
    lossfunction,
    optimizer,
    device,
    20
)

---Training Started---
Epoch 0: Training loss is 20.573692670976907, validation loss is 11.593443059488783.
Epoch 1: Training loss is 12.765696172603112, validation loss is 10.491174248295277.
Epoch 2: Training loss is 11.874472748128955, validation loss is 10.021790940156594.
Epoch 3: Training loss is 11.291439381461306, validation loss is 9.571192460582116.
Epoch 4: Training loss is 10.941847713116683, validation loss is 9.545986589714586.
Epoch 5: Training loss is 10.685662526928983, validation loss is 9.011685715574576.
Epoch 6: Training loss is 10.384545417995215, validation loss is 8.689210138600528.
Epoch 7: Training loss is 10.0972983795269, validation loss is 8.517024152868787.
Epoch 8: Training loss is 9.81557383451564, validation loss is 7.97970511636209.
Epoch 9: Training loss is 9.424773605470572, validation loss is 7.548356403406348.
Epoch 10: Training loss is 9.208628427613396, validation loss is 7.697611822053064.
Epoch 11: Training loss is 8.76705044639653, validation 

In [18]:
# Define full prediction pipeline with image transforms and model prediction
def predpipeline(imagepath):
  tensorimage = val_transform(Image.open(imagepath).convert("RGB"))
  tensorimage = tensorimage.to(device).unsqueeze(0)
  return Trained_Model(tensorimage)
# Predict the age of Yann Lecun (65 at the time of taking the photo)
print(predpipeline('yann.jpg'))

tensor([[73.0896]], device='cuda:0', grad_fn=<AddmmBackward0>)
